# Algorithm comparison: train QMIX (3 seeds) and evaluate

Settings: **Accelerator: None**, **Internet: On**. Run cells 1-2 to check, then **Save Version -> Save & Run All (Commit)** once (about 4-5 hours). Download **qmix_results.zip** from the Output tab.

In [ ]:
!pip install -q mujoco pettingzoo gymnasium
!git clone -q https://github.com/swarnasmita/marl-edu-platform.git
%cd marl-edu-platform
import os
print(os.cpu_count(), 'CPU cores')

In [ ]:
# quick check (about 1 minute): should end with a line starting with 'done;'
!python agents/train_qmix.py --seed 99 --out /kaggle/working/test_run --iterations 2 --episodes-per-iter 4 --eval-every 2 --val-scenarios 2 --workers 4
!rm -rf /kaggle/working/test_run

In [ ]:
# train three seeds (about 75 minutes each)
for seed in (0, 1, 2):
    !python agents/train_qmix.py --seed {seed} --out /kaggle/working/runs/qmix_seed{seed} --workers 4

In [ ]:
# evaluate the three seeds on the 100 unseen test scenarios x 3 simulated humans
!python agents/evaluate.py --workers 4 --label "QMIX (hierarchical)" --out /kaggle/working/results/algorithms/qmix --checkpoints /kaggle/working/runs/qmix_seed0/best.pt /kaggle/working/runs/qmix_seed1/best.pt /kaggle/working/runs/qmix_seed2/best.pt
!python agents/training_report.py /kaggle/working/runs/qmix_seed0 /kaggle/working/runs/qmix_seed1 /kaggle/working/runs/qmix_seed2
!mkdir -p /kaggle/working/results/algorithms/qmix && cp results/training_report.md /kaggle/working/results/algorithms/qmix/training_report.md && cp results/training_diagnostics.png /kaggle/working/results/algorithms/qmix/training_diagnostics.png

In [ ]:
# one zip to download
!cd /kaggle/working && zip -qr qmix_results.zip results runs/qmix_seed0/best.pt runs/qmix_seed1/best.pt runs/qmix_seed2/best.pt runs/qmix_seed0/log.csv runs/qmix_seed1/log.csv runs/qmix_seed2/log.csv
from IPython.display import Markdown, display
display(Markdown(open(f'/kaggle/working/results/algorithms/qmix/eval_table.md').read()))